In [16]:
import torch

In [17]:
torch.cuda.is_available()

True

In [18]:
from transformers import pipeline

MODEL_NAME = "tasksource/ModernBERT-base-nli"

modernbert_classifier = pipeline(
    task="zero-shot-classification",
    model=MODEL_NAME,
    device=0
)

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

In [19]:
import pandas as pd

DATA_PATH = "data/job_posts_evaluation.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (12, 2)


,job_description,expected_category
0,We need an engineer to build ETL pipelines usi...,data engineering and data pipelines
1,"Design dimensional models, maintain data wareh...",data engineering and data pipelines
2,Build machine learning models for text classif...,artificial intelligence and machine learning
3,"Develop RAG systems using embeddings, vector d...",artificial intelligence and machine learning
4,"Manage Kubernetes clusters, automate CI/CD pip...",devops and deployment automation


In [20]:
CATEGORIES = [
    "Data Engineering and Data Pipelines",
    "Artificial Intelligence and Machine Learning",
    "DevOps and Deployment Automation",
    "Cloud Infrastructure and Cloud Engineering",
    "Cybersecurity and Information Security",
    "Software and Application Development",
]

In [21]:
job_text = df.loc[0, "job_description"]

result = modernbert_classifier(
    job_text,
    candidate_labels=CATEGORIES,
    multi_label=False
)

print("Job Post:")
print(job_text)

print("\nPredicted Category:")
print(result["labels"][0])

print("\nConfidence:")
print(round(result["scores"][0], 4))

Job Post:
We need an engineer to build ETL pipelines using Python, SQL, Airflow, dbt, and Spark.

Predicted Category:
Data Engineering and Data Pipelines

Confidence:
0.45


In [22]:
result_df = pd.DataFrame({
    "Category": result["labels"],
    "Confidence": result["scores"]
})

result_df["Confidence"] = result_df["Confidence"].round(4)

result_df

,Category,Confidence
0,Data Engineering and Data Pipelines,0.4500
1,Software and Application Development,0.1798
2,Cloud Infrastructure and Cloud Engineering,0.1334
3,Artificial Intelligence and Machine Learning,0.0988
4,DevOps and Deployment Automation,0.0930
5,Cybersecurity and Information Security,0.0451


In [23]:
predictions = []
confidences = []

for job_text in df["job_description"]:
    result = modernbert_classifier(
        str(job_text),
        candidate_labels=CATEGORIES,
        multi_label=False
    )

    predictions.append(result["labels"][0])
    confidences.append(result["scores"][0])

In [24]:
df["modernbert_prediction"] = predictions
df["modernbert_confidence"] = confidences

df.head()

,job_description,expected_category,modernbert_prediction,modernbert_confidence
0,We need an engineer to build ETL pipelines usi...,data engineering and data pipelines,Data Engineering and Data Pipelines,0.450019
1,"Design dimensional models, maintain data wareh...",data engineering and data pipelines,Data Engineering and Data Pipelines,0.263233
2,Build machine learning models for text classif...,artificial intelligence and machine learning,Artificial Intelligence and Machine Learning,0.355238
3,"Develop RAG systems using embeddings, vector d...",artificial intelligence and machine learning,Software and Application Development,0.350874
4,"Manage Kubernetes clusters, automate CI/CD pip...",devops and deployment automation,DevOps and Deployment Automation,0.422626


In [25]:
import time

predictions = []
confidences = []
latencies = []

for index, row in df.iterrows():
    job_text = row["job_description"]

    start_time = time.perf_counter()

    result = modernbert_classifier(
        job_text,
        candidate_labels=CATEGORIES,
        multi_label=False
    )

    latency = time.perf_counter() - start_time

    predictions.append(result["labels"][0])
    confidences.append(result["scores"][0])
    latencies.append(latency)

    print(f"Processed {index + 1}/{len(df)}", end="\r")

print("\nEvaluation completed.")

Processed 12/12
Evaluation completed.


In [27]:
# إضافة التوقعات للداتا
df["predicted_category"] = predictions

# مقارنة التوقع بالإجابة الصحيحة
df["is_correct"] = (
    df["predicted_category"]
    == df["expected_category"]
)

# حساب الـ Accuracy
accuracy = df["is_correct"].mean()

# عدد الصح والغلط
correct_predictions = df["is_correct"].sum()
wrong_predictions = len(df) - correct_predictions

print(f"Accuracy: {accuracy:.2%}")
print(f"Correct predictions: {correct_predictions}")
print(f"Wrong predictions: {wrong_predictions}")

Accuracy: 0.00%
Correct predictions: 0
Wrong predictions: 12


In [26]:
modernbert_results = df.copy()

modernbert_results["predicted_category"] = predictions
modernbert_results["confidence"] = confidences
modernbert_results["latency_seconds"] = latencies

modernbert_results["is_correct"] = (
    modernbert_results["expected_category"]
    == modernbert_results["predicted_category"]
)

modernbert_results

,job_description,expected_category,modernbert_prediction,modernbert_confidence,predicted_category,confidence,latency_seconds,is_correct
0,We need an engineer to build ETL pipelines usi...,data engineering and data pipelines,Data Engineering and Data Pipelines,0.450019,Data Engineering and Data Pipelines,0.450019,0.236005,False
1,"Design dimensional models, maintain data wareh...",data engineering and data pipelines,Data Engineering and Data Pipelines,0.263233,Data Engineering and Data Pipelines,0.263233,0.089976,False
2,Build machine learning models for text classif...,artificial intelligence and machine learning,Artificial Intelligence and Machine Learning,0.355238,Artificial Intelligence and Machine Learning,0.355238,0.081366,False
3,"Develop RAG systems using embeddings, vector d...",artificial intelligence and machine learning,Software and Application Development,0.350874,Software and Application Development,0.350874,0.081878,False
4,"Manage Kubernetes clusters, automate CI/CD pip...",devops and deployment automation,DevOps and Deployment Automation,0.422626,DevOps and Deployment Automation,0.422626,0.076361,False
5,Create automated release pipelines using Docke...,devops and deployment automation,DevOps and Deployment Automation,0.339984,DevOps and Deployment Automation,0.339984,0.074011,False
6,"Design scalable AWS infrastructure using EC2, ...",cloud infrastructure and cloud engineering,Cloud Infrastructure and Cloud Engineering,0.369123,Cloud Infrastructure and Cloud Engineering,0.369123,0.093973,False
7,Build secure Azure cloud environments and mana...,cloud infrastructure and cloud engineering,Cybersecurity and Information Security,0.276002,Cybersecurity and Information Security,0.276002,0.091580,False
8,Monitor security events using SIEM tools and i...,cybersecurity and information security,Cybersecurity and Information Security,0.376939,Cybersecurity and Information Security,0.376939,0.088445,False
9,"Perform vulnerability assessments, threat dete...",cybersecurity and information security,Cybersecurity and Information Security,0.440240,Cybersecurity and Information Security,0.440240,0.084302,False


In [28]:
df["predicted_category"] = predictions

df[
    [
        "expected_category",
        "predicted_category"
    ]
]

,expected_category,predicted_category
0,data engineering and data pipelines,Data Engineering and Data Pipelines
1,data engineering and data pipelines,Data Engineering and Data Pipelines
2,artificial intelligence and machine learning,Artificial Intelligence and Machine Learning
3,artificial intelligence and machine learning,Software and Application Development
4,devops and deployment automation,DevOps and Deployment Automation
5,devops and deployment automation,DevOps and Deployment Automation
6,cloud infrastructure and cloud engineering,Cloud Infrastructure and Cloud Engineering
7,cloud infrastructure and cloud engineering,Cybersecurity and Information Security
8,cybersecurity and information security,Cybersecurity and Information Security
9,cybersecurity and information security,Cybersecurity and Information Security


In [29]:
df["predicted_category"] = predictions
df["confidence"] = confidences
df["latency_seconds"] = latencies

# تنظيف النصوص قبل المقارنة
df["expected_normalized"] = (
    df["expected_category"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["predicted_normalized"] = (
    df["predicted_category"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# المقارنة الصحيحة
df["is_correct"] = (
    df["expected_normalized"]
    == df["predicted_normalized"]
)

# الحسابات
accuracy = df["is_correct"].mean()
correct_predictions = int(df["is_correct"].sum())
wrong_predictions = len(df) - correct_predictions
average_latency = df["latency_seconds"].mean()

print(f"Accuracy: {accuracy:.2%}")
print(f"Correct predictions: {correct_predictions}")
print(f"Wrong predictions: {wrong_predictions}")
print(f"Average latency: {average_latency:.4f} seconds")

Accuracy: 83.33%
Correct predictions: 10
Wrong predictions: 2
Average latency: 0.0977 seconds


In [30]:
from pathlib import Path
import pandas as pd

# معلومات الموديل
MODEL_NAME = "ModernBERT"
MODEL_ID = "tasksource/ModernBERT-base-nli"

# التكلفة هنا صفر لأن التشغيل Local ومفيش API مدفوعة
COST_USD = 0.0
COST_DESCRIPTION = "Free - Local inference"

# مكان ملف المقارنة النهائي
RESULTS_FILE = Path("data/final_result_of_comparison.csv")

# إنشاء فولدر data لو مش موجود
RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)

# صف النتيجة الخاص بالموديل الحالي
new_result = pd.DataFrame([
    {
        "model_name": MODEL_NAME,
        "model_id": MODEL_ID,
        "accuracy": round(float(accuracy), 4),
        "average_latency_seconds": round(float(average_latency), 4),
        "cost_usd": COST_USD,
        "cost_description": COST_DESCRIPTION,
        "correct_predictions": int(correct_predictions),
        "wrong_predictions": int(wrong_predictions),
        "total_examples": int(len(df)),
    }
])

# لو الملف موجود، اقرأه وحدّث صف الموديل
if RESULTS_FILE.exists():
    comparison_df = pd.read_csv(RESULTS_FILE)

    # حذف النتيجة القديمة لنفس الموديل حتى لا يتكرر
    comparison_df = comparison_df[
        comparison_df["model_name"] != MODEL_NAME
    ]

    comparison_df = pd.concat(
        [comparison_df, new_result],
        ignore_index=True
    )
else:
    comparison_df = new_result

# ترتيب الموديلات من أعلى Accuracy
comparison_df = comparison_df.sort_values(
    by="accuracy",
    ascending=False
).reset_index(drop=True)

# حفظ الملف
comparison_df.to_csv(
    RESULTS_FILE,
    index=False
)

print(f"{MODEL_NAME} result saved successfully.")
print(f"Saved to: {RESULTS_FILE}")

comparison_df

ModernBERT result saved successfully.
Saved to: data\final_result_of_comparison.csv


,model_name,model_id,accuracy,average_latency_seconds,cost_usd,cost_description,correct_predictions,wrong_predictions,total_examples
0,ModernBERT,tasksource/ModernBERT-base-nli,0.8333,0.0977,0.0,Free - Local inference,10,2,12
